# U-Net 二段階学習（Google Colab）

1. **パッチ化** … 合成 1,000 + Guitar-TECHS
2. **Stage 1** … 合成データで学習
3. **Stage 2** … Guitar-TECHS でファインチューニング
4. **push** … `checkpoints/stage2/unet_last.pt` を GitHub へ

**事前設定（セル 1）**
- リポジトリが **Private** の場合: `GITHUB_TOKEN`（PAT, repo 権限）が **clone 時から必須**
- リポジトリが **Public** の場合: `GITHUB_TOKEN` は checkpoint push 時のみ必要

In [ ]:
REPO_URL = "https://github.com/Kakeru0307/colab_train.git"
GITHUB_TOKEN = ""  # ← Private リポジトリならここに PAT（repo 権限）を入れる

import os
if GITHUB_TOKEN:
    os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN

def authenticated_repo_url(repo_url: str, token: str) -> str:
    if not token:
        return repo_url
    if repo_url.startswith("https://") and "@" not in repo_url:
        return repo_url.replace("https://", f"https://{token}@")
    return repo_url

CLONE_URL = authenticated_repo_url(REPO_URL, GITHUB_TOKEN)
print("clone:", REPO_URL)
print("token:", "set" if GITHUB_TOKEN else "not set")

In [ ]:
import os
import shutil
from pathlib import Path

REPO_DIR = Path("repo")

# 作業ディレクトリを /content に戻す（再実行時の安全策）
if not Path("train.py").exists() and Path("/content").exists():
    os.chdir("/content")

if Path("train.py").exists():
    print("既にリポジトリ内です:", os.getcwd())
else:
    if REPO_DIR.exists() and not (REPO_DIR / "train.py").exists():
        print("不完全な repo を削除します")
        shutil.rmtree(REPO_DIR)

    if REPO_DIR.exists() and (REPO_DIR / "train.py").exists():
        print("既存の repo を使用します")
        os.chdir(REPO_DIR)
    else:
        if not GITHUB_TOKEN:
            print("警告: Private リポジトリの場合、セル1で GITHUB_TOKEN を設定してください")
        !git clone {CLONE_URL} repo
        os.chdir("repo")

    if not Path("train.py").exists():
        raise FileNotFoundError(
            "clone 失敗。GITHUB_TOKEN を設定するか、ランタイムを再起動して再実行してください。"
        )

print("OK:", os.getcwd())
!ls train.py prepare_all.py colab_train.ipynb

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
SYNTHETIC_COUNT = 1000
SYNTHETIC_SEED = 42

!python prepare_all.py --synthetic-count {SYNTHETIC_COUNT} --synthetic-seed {SYNTHETIC_SEED}

In [ ]:
STAGE1_EPOCHS = 20
STAGE2_EPOCHS = 20
BATCH_SIZE = 16

!python train.py \
  --pairs-dir data/pairs/synthetic \
  --epochs {STAGE1_EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --checkpoint-dir checkpoints/stage1 \
  --pos-weight 10

In [ ]:
!python train.py \
  --pairs-dir data/pairs/guitar-techs \
  --epochs {STAGE2_EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --lr 1e-5 \
  --checkpoint-dir checkpoints/stage2 \
  --resume checkpoints/stage1/unet_last.pt \
  --pos-weight 10

In [ ]:
!git config user.email "colab@users.noreply.github.com"
!git config user.name "Colab Train"
!python scripts/push_checkpoint.py \
  --ckpt checkpoints/stage2/unet_last.pt \
  --message "Stage1+2 train synthetic+techs"